# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent handles:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- Word/sentence statistics → Text Stats Tool *(bonus)*
- Current date/time → DateTime Tool *(bonus)*
- General queries → Direct response

---
### 🛠️ What's Implemented
- Agent logic with a central `agent()` entry point
- **Regex-based** conditional routing (not just plain substring matching, so `"80 + 90"` is
  correctly routed to the calculator even without the word "calculate")
- Tool integration (4 tools)
- Robust error handling at both the tool level and the agent level
- A **safe** calculator that parses expressions with `ast` instead of calling raw `eval()`

### 🚀 Bonus Features Implemented
- Improved routing (regex + intent detection function, easy to extend)
- Logging: every query/response is timestamped, logged via the `logging` module, and stored
  in an in-memory `agent_logs` list for auditing
- Two extra tools: **Text Stats** and **DateTime**
- Structured JSON output includes `type`, `tool_used`, `result`, `timestamp`, and `query`


## 📦 Imports & Logging Setup

In [1]:
import re
import json
import logging
from datetime import datetime

# ---- Logging setup ----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("SmartAgent")

# In-memory log of every agent call (bonus: simple audit trail)
agent_logs = []


## 🛠️ TOOL 1: Calculator (safe — no raw `eval`)

In [2]:
import ast
import operator

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed")
    if isinstance(node, ast.BinOp):
        op_type = type(node.op)
        if op_type not in _ALLOWED_OPERATORS:
            raise ValueError(f"Unsupported operator: {op_type.__name__}")
        return _ALLOWED_OPERATORS[op_type](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp):
        op_type = type(node.op)
        if op_type not in _ALLOWED_OPERATORS:
            raise ValueError(f"Unsupported operator: {op_type.__name__}")
        return _ALLOWED_OPERATORS[op_type](_eval_node(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression: str) -> str:
    '''Safely evaluate a mathematical expression (uses ast parsing, not eval()).'''
    expression = expression.strip()
    if not expression:
        return "Error: empty expression"
    try:
        tree = ast.parse(expression, mode="eval")
        result = _eval_node(tree.body)
        return str(result)
    except ZeroDivisionError:
        return "Error: division by zero"
    except Exception as e:
        return f"Error in calculation: {e}"


## 🛠️ TOOL 2: Keyword Extractor (stopword-aware)

In [3]:
STOPWORDS = {
    "this", "that", "with", "from", "have", "about", "which", "there",
    "their", "would", "could", "should", "these", "those", "being",
    "where", "after", "before", "because", "while", "extract",
}


def extract_keywords(text: str, top_n: int = 5) -> list:
    '''Extract simple keywords from text, filtering stopwords/punctuation/duplicates.'''
    try:
        words = re.findall(r"[A-Za-z']+", text)
        keywords, seen = [], set()
        for w in words:
            wl = w.lower()
            if len(wl) > 4 and wl not in STOPWORDS and wl not in seen:
                seen.add(wl)
                keywords.append(wl)
        return keywords[:top_n]
    except Exception:
        return []


## 🛠️ TOOL 3 (bonus): Text Stats

In [4]:
def text_stats(text: str) -> dict:
    '''Return word / character / sentence counts for a piece of text.'''
    try:
        words = text.split()
        chars = len(text)
        sentence_count = len(re.findall(r"[.!?]+", text)) or (1 if text.strip() else 0)
        return {"words": len(words), "characters": chars, "sentences": sentence_count}
    except Exception:
        return {}


## 🛠️ TOOL 4 (bonus): DateTime

In [5]:
def get_datetime(_: str = "") -> str:
    '''Return the current date and time.'''
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


## 🤖 Agent Logic

**Routing rules (bonus: improved routing):**
- Contains "keyword(s)" → Keyword Extractor
- Asks about word/character/sentence counts → Text Stats
- Asks for the current time/date → DateTime tool
- Contains "calculate"/"compute", **or** looks like a bare math expression
  (e.g. `"80 + 90"`) → Calculator
- Anything else → General direct response

Every call is logged (timestamp + intent + response) and appended to `agent_logs`.

In [6]:
MATH_EXPRESSION_PATTERN = re.compile(r"^[\d\s\+\-\*/%\.\(\)]+$")


def detect_intent(query: str) -> str:
    q = query.lower().strip()

    if "keyword" in q:
        return "keywords"
    if any(p in q for p in ["word count", "count words", "how many words", "text stats"]):
        return "text_stats"
    if "time" in q or "date" in q:
        return "datetime"
    if "calculate" in q or "compute" in q or MATH_EXPRESSION_PATTERN.match(q.strip()):
        return "calculation"
    return "general"


def agent(query: str) -> dict:
    '''Single-agent entry point: routes the query, calls the right tool, returns structured JSON.'''
    timestamp = datetime.now().isoformat()
    intent = detect_intent(query)
    logger.info(f"Received query: '{query}' | Detected intent: {intent}")

    response = {
        "query": query,
        "type": intent,
        "tool_used": None,
        "result": None,
        "timestamp": timestamp,
    }

    try:
        if intent == "calculation":
            expression = re.sub(r"calculate|compute", "", query, flags=re.IGNORECASE).strip()
            if not expression:
                response["type"] = "error"
                response["result"] = "No mathematical expression provided."
            else:
                response["tool_used"] = "calculator"
                response["result"] = calculator(expression)

        elif intent == "keywords":
            text = re.sub(r"extract keywords from|keywords", "", query, flags=re.IGNORECASE).strip()
            if not text:
                response["type"] = "error"
                response["result"] = "No text provided for keyword extraction."
            else:
                response["tool_used"] = "keyword_extractor"
                response["result"] = extract_keywords(text)

        elif intent == "text_stats":
            response["tool_used"] = "text_stats"
            response["result"] = text_stats(query)

        elif intent == "datetime":
            response["tool_used"] = "datetime_tool"
            response["result"] = get_datetime()

        else:
            response["type"] = "general"
            response["result"] = f"I understand your question: '{query}'. This doesn't require any tool."

    except Exception as e:
        logger.error(f"Error processing query '{query}': {e}")
        response["type"] = "error"
        response["result"] = str(e)

    agent_logs.append(response)
    logger.info(f"Response: {json.dumps(response)}")
    return response


## 📦 Expected Output Format

```
{
  "query": "...",
  "type": "calculation / keywords / text_stats / datetime / general / error",
  "tool_used": "calculator / keyword_extractor / text_stats / datetime_tool / null",
  "result": ...,
  "timestamp": "..."
}
```

## 🧪 Test Cases

In [7]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "80 + 90",                       # math expression without the word "calculate"
    "calculate 10 / 0",               # error handling: division by zero
    "calculate",                      # error handling: no expression given
    "How many words are in this sentence right now",
    "What is the current time?",
]

for q in queries:
    resp = agent(q)
    print("Query:", q)
    print("Response:", json.dumps(resp, indent=2))
    print("-" * 60)


16:42:11 | INFO | Received query: 'Calculate 20 + 5' | Detected intent: calculation


16:42:11 | INFO | Response: {"query": "Calculate 20 + 5", "type": "calculation", "tool_used": "calculator", "result": "25", "timestamp": "2026-07-12T16:42:11.917437"}


16:42:11 | INFO | Received query: 'Extract keywords from Artificial Intelligence is transforming industries' | Detected intent: keywords


16:42:11 | INFO | Response: {"query": "Extract keywords from Artificial Intelligence is transforming industries", "type": "keywords", "tool_used": "keyword_extractor", "result": ["artificial", "intelligence", "transforming", "industries"], "timestamp": "2026-07-12T16:42:11.920673"}


16:42:11 | INFO | Received query: 'What is machine learning?' | Detected intent: general


...

16:42:11 | INFO | Received query: 'How many words are in this sentence right now' | Detected intent: text_stats


16:42:11 | INFO | Response: {"query": "How many words are in this sentence right now", "type": "text_stats", "tool_used": "text_stats", "result": {"words": 9, "characters": 45, "sentences": 1}, "timestamp": "2026-07-12T16:42:11.928373"}


16:42:11 | INFO | Received query: 'What is the current time?' | Detected intent: datetime


16:42:11 | INFO | Response: {"query": "What is the current time?", "type": "datetime", "tool_used": "datetime_tool", "result": "2026-07-12 16:42:11", "timestamp": "2026-07-12T16:42:11.929793"}


Query: Calculate 20 + 5
Response: {
  "query": "Calculate 20 + 5",
  "type": "calculation",
  "tool_used": "calculator",
  "result": "25",
  "timestamp": "2026-07-12T16:42:11.917437"
}
------------------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {
  "query": "Extract keywords from Artificial Intelligence is transforming industries",
  "type": "keywords",
  "tool_used": "keyword_extractor",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ],
  "timestamp": "2026-07-12T16:42:11.920673"
}
------------------------------------------------------------
Query: What is machine learning?
Response: {
  "query": "What is machine learning?",
  "type": "general",
  "tool_used": null,
  "result": "I understand your question: 'What is machine learning?'. This doesn't require any tool.",
  "timestamp": "2026-07-12T16:42:11.922717"
}
----------------------------------------

## 📊 Log Summary (bonus)

In [8]:
print(f"Total queries processed: {len(agent_logs)}")
type_counts = {}
for entry in agent_logs:
    type_counts[entry["type"]] = type_counts.get(entry["type"], 0) + 1

print("Breakdown by type:")
for t, c in type_counts.items():
    print(f"  {t}: {c}")


Total queries processed: 8
Breakdown by type:
  calculation: 3
  keywords: 1
  general: 1
  error: 1
  text_stats: 1
  datetime: 1


## 🎯 Interactive Mode

Run this cell and type queries at the prompt. Type `exit` to stop.

In [ ]:
while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", json.dumps(agent(user_input), indent=2))
